In [12]:
# 약속 1. Data 규칙 부여 시 min, max etc. 는 _min = / _max 와 같이 사용할 것
# ex) data_***_min = data_rate.min()

#약속 2. 약어를 쓸 경우 각 단어의 앞 글자로만 구성할 것.
# Released_Year = ry

#약속 3. Extract = ext, Seperate = spr, Filter = flt etc. 단어가 길 경우 앞의 모음 세 글자를 사용하는 것으로 약속 만약 겹칠 경우 끝자리를 달리하는 등의 규칙을 여기에 꼭 표기할 것.
#  Extend  = exd (추가), director = drc (추가)

#약속 4. 수정사항은 # 표기와 함께 간략히 적을 것 (ex. 명칭 변경, 숫자 변경 등)

import numpy as np
import csv

with open ('IMDB top 1000.csv','r',encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)
    data_list=[row for row in reader]

data_array = np.array(data_list)

#2-1
# 컬럼명 순서 : ,Title,Certificate,Duration,Genre,Rate,Metascore,Description,Cast,Info
data_title = data_array[:,1:2]
data_genre = data_array[:,4:5]
data_rate = data_array[:,5:6]
#Checker
#print(data_title[:3])
#print(data_genre[:3])
#print(data_rate[:3]) OK

#2-2 Released_Year 슬라이싱 뒤의 여섯자리 출력 [list comprehesion] - *Step 5에서 사용*
released_year = np.array([year[0][-5:-1] for year in data_title])
data_ry = released_year.reshape(-1,1) #array 배열로 변경
#print(data_ry) OK

#3 Numpy 배열로 변환
data_stack = np.column_stack((data_title, data_genre, data_rate,data_ry))
column_header_step1 = "Title,Genre,Rate,Released_Year"
data_stack_array = np.array(data_stack)
#print(data_stack) OK

#4 결측값이 있는 행을 제거하세요. csv.reader의 결측값은 empty space
# 한 개라도 결측치가 있을 경우 row 제거 = np.all() / row 기준 axis=1
fil_NaN = np.all(data_stack_array != '', axis=1)
data_filter = data_stack_array[fil_NaN]
data_step1 = data_filter
#print(data_step1)

#np.savetxt(
#    "IMDB top 1000 R3.csv",
#    data_step1,delimiter=',',
#    fmt="%s",
#    header=column_header_step1,
#    comments="",encoding='utf-8')

#5 Cast_Director 분리
data_cast = data_array[:,8:9]
director_spe = np.char.split(data_cast[:,0],' |')
data_director = np.array([drc_row[0].replace('Director: ','') for drc_row in director_spe])
data_director = data_director.reshape(-1,1)
#print(data_director)


#Step4

#4-1 필요한 열 추출 및 타입 지정
data_gnr = data_step1[:,1].astype(str)
data_rte = data_step1[:,2].astype(float)

#4-2 장르 분리 및 리스트에 모으기
gnr_all_lst = []
for gnr_row in data_gnr:
    gnr_spr = [gnr_itm.strip() for gnr_itm in gnr_row.split(",")]
    gnr_all_lst.extend(gnr_spr)

#4-3 장르 중복 제거
data_gnr_all = np.array(gnr_all_lst, dtype=object)
data_gnr_uni = np.unique(data_gnr_all)

#4-4 장르별 평균 평점 계산
gnr_avg_lst = []
for gnr_itm in data_gnr_uni:
    data_gnr_flt = (np.char.find(data_gnr,gnr_itm) >= 0)
    data_rte_avg = data_rte[data_gnr_flt].mean()
    gnr_avg_lst.append((gnr_itm, data_rte_avg))
data_gnr_avg = np.array(gnr_avg_lst, dtype=object)

#4-5 평균 평점 기준으로 정렬
data_rte_avg_val = data_gnr_avg[:,1].astype(float)
data_rte_avg_ord = np.argsort(data_rte_avg_val)[::-1]
data_gnr_avg_srt = data_gnr_avg[data_rte_avg_ord]

print("장르별 평균 평점:")
for gnr_itm, rte_avg in data_gnr_avg_srt:
    print(f"{gnr_itm}: {float(rte_avg):.2f}")



print("-"*40)



#Step 4 보너스로 장르별 작품수 확인
# 1. Step 4-2에서 만든 gnr_all_lst를 활용 (모든 영화의 장르가 분리되어 담긴 리스트)
data_gnr_all = np.array(gnr_all_lst, dtype=object)

# 2. NumPy의 unique 함수로 장르명과 등장 횟수를 한 번에 추출
genres, counts = np.unique(data_gnr_all, return_counts=True)

# 3. (장르, 개수) 형태로 묶기
gnr_counts = np.array(list(zip(genres, counts)), dtype=object)

# 4. 작품 수가 많은 순서대로 정렬 (내림차순)
# counts(컬럼 인덱스 1)를 기준으로 정렬
sorted_indices = np.argsort(gnr_counts[:, 1].astype(int))[::-1]
gnr_counts_sorted = gnr_counts[sorted_indices]

# 5. 결과 출력
print("장르별 작품 수 (TOP 1000 내 빈도):")
for gnr, cnt in gnr_counts_sorted:
    print(f"{gnr}: {cnt}편")



장르별 평균 평점:
Western: 8.36
War: 8.21
Horror: 8.19
Sport: 8.18
Family: 8.16
Film-Noir: 8.13
Crime: 8.12
Fantasy: 8.11
Mystery: 8.11
Sci-Fi: 8.11
Drama: 8.10
History: 8.09
Adventure: 8.09
Musical: 8.09
Action: 8.09
Music: 8.08
Romance: 8.08
Comedy: 8.07
Thriller: 8.07
Biography: 8.07
Animation: 8.06
----------------------------------------
장르별 작품 수 (TOP 1000 내 빈도):
Drama: 728편
Adventure: 211편
Comedy: 208편
Action: 207편
Crime: 176편
Thriller: 165편
Romance: 147편
Biography: 129편
Mystery: 103편
Animation: 100편
History: 66편
Sci-Fi: 63편
Fantasy: 53편
War: 37편
Family: 30편
Music: 28편
Film-Noir: 17편
Horror: 13편
Musical: 12편
Sport: 8편
Western: 7편
